# Feature Engineering for Concrete Compressive Strength

This notebook creates domain-informed features, compares six feature sets with fold-safe pipelines, and saves the final engineered training artifacts. Scaling is intentionally deferred until after model and feature-set selection.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data" / "interim" / "X_train.csv").exists()
)

INTERIM_PATH = PROJECT_ROOT / "data" / "interim"
PREPROCESSING_PATH = PROJECT_ROOT / "models" / "preprocessing"
REPORT_PATH = PROJECT_ROOT / "reports" / "analysis"
FIGURES_PATH = PROJECT_ROOT / "reports" / "figures"
for output_path in (PREPROCESSING_PATH, REPORT_PATH, FIGURES_PATH):
    output_path.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(INTERIM_PATH / "X_train.csv")
X_test = pd.read_csv(INTERIM_PATH / "X_test.csv")
y_train = pd.read_csv(INTERIM_PATH / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(INTERIM_PATH / "y_test.csv").squeeze("columns")

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Test target shape: {y_test.shape}")

## 1. Validate the preprocessing contract

In [ ]:
expected_columns = [
    "cement", "blast_furnace_slag", "fly_ash", "water",
    "superplasticizer", "coarse_aggregate", "fine_aggregate", "age",
]

assert list(X_train.columns) == expected_columns
assert list(X_test.columns) == expected_columns
assert len(X_train) == len(y_train) == 804
assert len(X_test) == len(y_test) == 201
assert X_train.notna().all().all() and X_test.notna().all().all()
assert y_train.notna().all() and y_test.notna().all()
assert not np.isinf(X_train.to_numpy()).any()
assert not np.isinf(X_test.to_numpy()).any()
print("Input contract validation passed.")

## 2. Define leakage-safe feature engineering

In [ ]:
def engineer_features(frame):
    engineered = frame.copy()
    epsilon = 1e-8
    engineered["total_binder"] = engineered["cement"] + engineered["blast_furnace_slag"] + engineered["fly_ash"]
    engineered["total_aggregate"] = engineered["coarse_aggregate"] + engineered["fine_aggregate"]
    engineered["total_mix_material"] = engineered["total_binder"] + engineered["water"] + engineered["superplasticizer"] + engineered["total_aggregate"]
    engineered["water_cement_ratio"] = engineered["water"] / (engineered["cement"] + epsilon)
    engineered["water_binder_ratio"] = engineered["water"] / (engineered["total_binder"] + epsilon)
    engineered["cement_binder_ratio"] = engineered["cement"] / (engineered["total_binder"] + epsilon)
    engineered["slag_binder_ratio"] = engineered["blast_furnace_slag"] / (engineered["total_binder"] + epsilon)
    engineered["fly_ash_binder_ratio"] = engineered["fly_ash"] / (engineered["total_binder"] + epsilon)
    engineered["superplasticizer_binder_ratio"] = engineered["superplasticizer"] / (engineered["total_binder"] + epsilon)
    engineered["aggregate_binder_ratio"] = engineered["total_aggregate"] / (engineered["total_binder"] + epsilon)
    engineered["water_total_mix_ratio"] = engineered["water"] / (engineered["total_mix_material"] + epsilon)
    engineered["log_age"] = np.log1p(engineered["age"])
    engineered["sqrt_age"] = np.sqrt(engineered["age"])
    engineered["age_squared"] = engineered["age"] ** 2
    engineered["cement_x_age"] = engineered["cement"] * engineered["age"]
    engineered["binder_x_age"] = engineered["total_binder"] * engineered["age"]
    engineered["water_x_age"] = engineered["water"] * engineered["age"]
    engineered["water_cement_ratio_x_age"] = engineered["water_cement_ratio"] * engineered["age"]
    engineered["superplasticizer_x_binder_ratio"] = engineered["superplasticizer"] * engineered["superplasticizer_binder_ratio"]
    return engineered.replace([np.inf, -np.inf], np.nan)

X_train_engineered = engineer_features(X_train)
X_test_engineered = engineer_features(X_test)
assert list(X_train_engineered.columns) == list(X_test_engineered.columns)
assert not X_train_engineered.isna().any().any()
assert not X_test_engineered.isna().any().any()
assert np.isfinite(X_train_engineered.to_numpy()).all()
assert np.isfinite(X_test_engineered.to_numpy()).all()
print(f"Engineered feature count: {X_train_engineered.shape[1]}")

## 3. Define six feature sets

In [ ]:
ratio_features = ["water_cement_ratio", "water_binder_ratio", "cement_binder_ratio", "slag_binder_ratio", "fly_ash_binder_ratio", "superplasticizer_binder_ratio", "aggregate_binder_ratio", "water_total_mix_ratio"]
total_features = ["total_binder", "total_aggregate", "total_mix_material"]
age_features = ["log_age", "sqrt_age", "age_squared"]
interaction_features = ["cement_x_age", "binder_x_age", "water_x_age", "water_cement_ratio_x_age", "superplasticizer_x_binder_ratio"]

feature_sets = {
    "original": expected_columns,
    "ratios": expected_columns + ratio_features,
    "totals": expected_columns + total_features,
    "age_transformations": expected_columns + age_features,
    "interactions": expected_columns + interaction_features,
    "all_engineered": list(X_train_engineered.columns),
}

for feature_set_name, columns in feature_sets.items():
    assert len(columns) == len(set(columns)), feature_set_name
    assert set(columns).issubset(X_train_engineered.columns), feature_set_name

for feature_set_name, columns in feature_sets.items():
    print(f"{feature_set_name}: {len(columns)} features")

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "gradient_boosting": GradientBoostingRegressor(random_state=42),
}
scoring = {
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "rmse": make_scorer(lambda actual, predicted: np.sqrt(mean_squared_error(actual, predicted)), greater_is_better=False),
    "r2": make_scorer(r2_score),
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
comparison_rows = []

for feature_set_name, columns in feature_sets.items():
    for model_name, model in models.items():
        pipeline = Pipeline([("scaler", StandardScaler()), ("model", model)])
        scores = cross_validate(pipeline, X_train_engineered[columns], y_train, cv=cv, scoring=scoring, n_jobs=1)
        comparison_rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "mae": -scores["test_mae"].mean(),
            "rmse": -scores["test_rmse"].mean(),
            "r2": scores["test_r2"].mean(),
            "mae_std": scores["test_mae"].std(),
            "rmse_std": scores["test_rmse"].std(),
            "r2_std": scores["test_r2"].std(),
        })

comparison_results = pd.DataFrame(comparison_rows).sort_values(["rmse", "mae"]).reset_index(drop=True)
print(comparison_results.to_string(index=False))
print("All comparisons used the same shuffled 5-fold splitter and fold-local scaler.")

## 5. Select the winning feature set and model

In [ ]:
winning_row = comparison_results.iloc[0]
winning_feature_set = winning_row["feature_set"]
winning_model_name = winning_row["model"]
winning_columns = feature_sets[winning_feature_set]
print(f"Winning feature set: {winning_feature_set}")
print(f"Winning model: {winning_model_name}")
print(f"Cross-validated RMSE: {winning_row['rmse']:.4f}")
print(f"Cross-validated MAE: {winning_row['mae']:.4f}")
print(f"Cross-validated R2: {winning_row['r2']:.4f}")

## 6. Fit and save the final feature scaler

The scaler is fitted only after model and feature-set selection, using the full engineered training data.

In [ ]:
final_scaler = StandardScaler()
X_train_final = X_train_engineered[winning_columns].copy()
X_test_final = X_test_engineered[winning_columns].copy()
X_train_final_scaled = final_scaler.fit_transform(X_train_final)
X_test_final_scaled = final_scaler.transform(X_test_final)

assert np.allclose(X_train_final_scaled.mean(axis=0), 0.0, atol=1e-10)
assert np.allclose(X_train_final_scaled.std(axis=0), 1.0, atol=1e-10)
assert np.isfinite(X_train_final_scaled).all()
assert np.isfinite(X_test_final_scaled).all()

scaled_train_frame = pd.DataFrame(X_train_final_scaled, columns=winning_columns)
scaled_test_frame = pd.DataFrame(X_test_final_scaled, columns=winning_columns)
X_train_final.to_csv(INTERIM_PATH / "X_train_engineered.csv", index=False)
X_test_final.to_csv(INTERIM_PATH / "X_test_engineered.csv", index=False)
scaled_train_frame.to_csv(INTERIM_PATH / "X_train_engineered_scaled.csv", index=False)
scaled_test_frame.to_csv(INTERIM_PATH / "X_test_engineered_scaled.csv", index=False)
comparison_results.to_csv(REPORT_PATH / "feature_set_comparison.csv", index=False)
joblib.dump(final_scaler, PREPROCESSING_PATH / "feature_scaler.joblib")

feature_engineering_report = {
    "input_train_rows": int(len(X_train)),
    "input_test_rows": int(len(X_test)),
    "original_feature_count": int(len(expected_columns)),
    "engineered_feature_count": int(X_train_engineered.shape[1]),
    "selected_feature_set": winning_feature_set,
    "selected_model": winning_model_name,
    "selected_features": winning_columns,
    "cross_validation": {"method": "KFold", "n_splits": 5, "shuffle": True, "random_state": 42, "scaling": "StandardScaler refit inside each pipeline fold"},
    "final_scaling": {"fit_data": "full engineered training data", "scaler_path": str((PREPROCESSING_PATH / "feature_scaler.joblib").relative_to(PROJECT_ROOT))},
    "feature_groups": {"ratio_features": ratio_features, "total_features": total_features, "age_features": age_features, "interaction_features": interaction_features},
    "artifacts": {
        "engineered_train": str((INTERIM_PATH / "X_train_engineered.csv").relative_to(PROJECT_ROOT)),
        "engineered_test": str((INTERIM_PATH / "X_test_engineered.csv").relative_to(PROJECT_ROOT)),
        "scaled_engineered_train": str((INTERIM_PATH / "X_train_engineered_scaled.csv").relative_to(PROJECT_ROOT)),
        "scaled_engineered_test": str((INTERIM_PATH / "X_test_engineered_scaled.csv").relative_to(PROJECT_ROOT)),
        "comparison_results": str((REPORT_PATH / "feature_set_comparison.csv").relative_to(PROJECT_ROOT)),
    },
}

with open(REPORT_PATH / "feature_engineering_report.json", "w", encoding="utf-8") as report_file:
    json.dump(feature_engineering_report, report_file, indent=2)

print("Final feature scaling validation passed.")
print(f"Final feature count: {len(winning_columns)}")
print(f"Selected feature set: {winning_feature_set}")
print(f"Feature scaler saved to: {PREPROCESSING_PATH / 'feature_scaler.joblib'}")
print(f"Report saved to: {REPORT_PATH / 'feature_engineering_report.json'}")